






















## Seguim intentant trobar una bona distribució de T respecte TSV i TCV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


# Put your notebook path here once (or set it via VS Code explorer)
NOTEBOOK_DIR = Path(r"C:\Users\labor\Documents\GitHub\TFM-Thermal-Walks\statistical_try")  # <-- change this

out_dir = NOTEBOOK_DIR / "figures"
out_dir.mkdir(parents=True, exist_ok=True)




In [ ]:
df_votes = pd.read_csv(r"../../data/all_surveys(votes).csv")
print(df_votes.columns)
df_votes["thermal_comfort"].unique


## Primer farem un scatter de TCV (T)

In [ ]:
import matplotlib.pyplot as plt



# choose the two columns
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"
tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}


df_plot = df_votes.copy()
df_plot["tcv_num"] = df_plot[y_col].map(tcv_map)
df_plot[x_col] = pd.to_numeric(df_plot[x_col], errors="coerce")
df_plot = df_plot.dropna(subset=[x_col, "tcv_num"]).sort_values(x_col)

ax = df_plot.plot.scatter(x=x_col, y="tcv_num")

# y-axis: ordered 1..7, but show labels (not numbers)
order = [k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])]
ax.set_yticks(range(1, 8))
ax.set_yticklabels(order)
ax.set_ylim(0.5, 7.5)

ax.set_ylabel("Thermal Comfort Vote")
ax.set_xlabel("Corrected Temperature (°C)")
plt.show()

# pooled distribution

In [ ]:
# Primera pooled distribution of general votes
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np

x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

# keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# ensure numeric x
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df = plot_df.dropna(subset=[x_col])

# optional: keep only valid thermal comfort categories
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=["TCV"])

# data
x = plot_df[x_col].to_numpy()

# KDE smooth density
kde = gaussian_kde(x, bw_method=0.30)
x_grid = np.linspace(x.min(), x.max(), 500)
density = kde(x_grid)

plt.figure(figsize=(8, 4.5))

plt.plot(
    x_grid,
    density,
    linewidth=2,
    label="KDE density",
)

plt.hist(
    x,
    bins=20,
    density=True,
    alpha=0.30,
    edgecolor="black",
    label="Histogram density",
)

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Density")
plt.title("Smooth pooled distribution of votes by T")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Primera pooled distribution of general votes
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np

x_col = "<HDX-HDX_fixed+<HDX>>"
y_col = "thermal_comfort"

# keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# ensure numeric x
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df = plot_df.dropna(subset=[x_col])

# optional: keep only valid thermal comfort categories
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=["TCV"])

# data
x = plot_df[x_col].to_numpy()

# KDE smooth density
kde = gaussian_kde(x, bw_method=0.30)
x_grid = np.linspace(x.min(), x.max(), 500)
density = kde(x_grid)

plt.figure(figsize=(8, 4.5))

plt.plot(
    x_grid,
    density,
    linewidth=2,
    label="KDE density",
)

plt.hist(
    x,
    bins=20,
    density=True,
    alpha=0.30,
    edgecolor="black",
    label="Histogram density",
)

plt.xlabel("Corrected HDX ")
plt.ylabel("Density")
plt.title("Smooth pooled distribution of votes by corrected HDX")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## No ens ensenya absolutament res tot i que sembla prou coherent? Molaria centrar-ho amb $\Delta T/T$ del thermal walk

# MEAN T x TSV-> Ara MEAN T. Després farem al revés (MEAN TSV(t BIN))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

# keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# ensure numeric x
plot_df = plot_df.dropna(subset=[x_col])

# map thermal comfort to numeric TCV (1..7)
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=["TCV"])

# group by category and compute temperature statistics
summary = (
    plot_df.groupby([y_col, "TCV"], as_index=False)[x_col]
    .agg(mean_T="mean", std_T="std", n="count")
    .sort_values("TCV")
)

# standard error
summary["sem_T"] = summary["std_T"] / summary["n"].pow(0.5)

print(summary)

# Plot: mean temperature vs TCV category, with horizontal error bars
plt.figure(figsize=(7, 5))
plt.errorbar(
    summary["mean_T"],
    summary["TCV"],
    xerr=summary["sem_T"],  # or summary["std_T"]
    fmt="o",
    capsize=4,
)

# y axis ordered 1..7 with labels (not numbers)
ordered_labels = [k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])]
plt.yticks(range(1, 8), ordered_labels)
plt.ylim(0.5, 7.5)

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Thermal Comfort")
plt.title("Thermal Comfort Vote vs Corrected Temperature")
plt.grid(True, alpha=0.3)
plt.show()

summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

# Keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# Ensure numeric temperature
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df = plot_df.dropna(subset=[x_col])

# Map thermal comfort to numeric order
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=["TCV"])

# Ordered labels from TCV 1 to 7
ordered_labels = [
    k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])
]

# Data in correct order
box_data = [
    plot_df.loc[plot_df["TCV"] == i, x_col].values
    for i in range(1, 8)
]

# Plot
plt.figure(figsize=(9, 5))

plt.boxplot(
    box_data,
    vert=False,
    labels=ordered_labels,
    showmeans=True,
    meanline=True,
    patch_artist=False,
    medianprops={
        "color": "black",
        "linewidth": 2,
    },
    meanprops={
        "color": "red",
        "linewidth": 2,
        "linestyle": "--",
    },
)

plt.xticks(rotation=0, ha="right")

plt.ylabel("Thermal Comfort")
plt.xlabel("Corrected Temperature (°C)")
plt.title("Corrected Temperature Distribution by Thermal Comfort Vote")
plt.grid(True, axis="y", alpha=0.3)
plt.legend(["Mean","Median"], loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

x_col = "<HDX-HDX_fixed+<HDX>>"
y_col = "thermal_comfort"

# Keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# Ensure numeric temperature
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df = plot_df.dropna(subset=[x_col])

# Map thermal comfort to numeric order
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=["TCV"])

# Ordered labels from TCV 1 to 7
ordered_labels = [
    k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])
]

# Data in correct order
box_data = [
    plot_df.loc[plot_df["TCV"] == i, x_col].values
    for i in range(1, 8)
]

# Plot
plt.figure(figsize=(9, 5))

plt.boxplot(
    box_data,
    vert=False,
    labels=ordered_labels,
    showmeans=True,
    meanline=True,
    patch_artist=False,
    medianprops={
        "color": "black",
        "linewidth": 2,
    },
    meanprops={
        "color": "red",
        "linewidth": 2,
        "linestyle": "--",
    },
)

plt.xticks(rotation=0, ha="right")

plt.ylabel("Thermal Comfort")
plt.xlabel("Corrected HDX")
plt.title("Corrected HDX Distribution by Thermal Comfort Vote")
plt.grid(True, axis="y", alpha=0.3)
plt.legend(["Mean","Median"], loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

summary = summary.sort_values("TCV").reset_index(drop=True).copy()

summary["ci95_T"] = t.ppf(0.975, summary["n"] - 1) * summary["sem_T"]

# Neutral is included in BOTH sides
comfortable = summary[summary["TCV"].between(1, 4)].copy()
uncomfortable = summary[summary["TCV"].between(4, 7)].copy()

ordered_labels = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]


def fit_temperature_as_function_of_tcv(data, error_col):
    """
    Fits:
        Temperature = m * TCV + b

    m = °C per TCV category
    """
    tcv = data["TCV"].to_numpy(float)
    temp = data["mean_T"].to_numpy(float)
    sigma = data[error_col].to_numpy(float)

    m, b = np.polyfit(
        tcv,
        temp,
        deg=1,
        w=1 / sigma
    )

    return m, b


def intersection(m1, b1, m2, b2):
    """
    Intersection between:
        T = m1 * TCV + b1
        T = m2 * TCV + b2
    """
    tcv_star = (b2 - b1) / (m1 - m2)
    temp_star = m1 * tcv_star + b1
    return temp_star, tcv_star


def run_case(error_col, label):
    m_c, b_c = fit_temperature_as_function_of_tcv(comfortable, error_col)
    m_u, b_u = fit_temperature_as_function_of_tcv(uncomfortable, error_col)

    temp_star, tcv_star = intersection(m_c, b_c, m_u, b_u)

    print("\n" + label.upper())
    print("-" * len(label))

    print("\nComfortable side: Very comfortable → Neutral")
    print(f"T = {m_c:.5f} * TCV + {b_c:.5f}")
    print(f"°C per TCV = {m_c:.5f}")
    print(f"TCV per °C = {1 / m_c:.5f}")

    print("\nUncomfortable side: Neutral → Very uncomfortable")
    print(f"T = {m_u:.5f} * TCV + {b_u:.5f}")
    print(f"°C per TCV = {m_u:.5f}")
    print(f"TCV per °C = {1 / m_u:.5f}")

    print("\nIntersection")
    print(f"T = {temp_star:.3f} °C")
    print(f"TCV = {tcv_star:.3f}")

    plt.figure(figsize=(8, 5))

    # Data points with horizontal uncertainty
    plt.errorbar(
        summary["mean_T"],
        summary["TCV"],
        xerr=summary[error_col],
        fmt="o",
        capsize=4,
        label=f"Mean ± {label}",
    )

    # Comfortable fit: TCV 1 to 4
    tcv_c = np.linspace(1, 4, 100)
    temp_c = m_c * tcv_c + b_c

    plt.plot(
        temp_c,
        tcv_c,
        label="Fit: Very comfortable → Neutral",
    )

    # Uncomfortable fit: TCV 4 to 7
    tcv_u = np.linspace(4, 7, 100)
    temp_u = m_u * tcv_u + b_u

    plt.plot(
        temp_u,
        tcv_u,
        label="Fit: Neutral → Very uncomfortable",
    )

    # Intersection
    plt.scatter(
        temp_star,
        tcv_star,
        marker="x",
        s=130,
        label=f"Intersection: {temp_star:.2f} °C, TCV={tcv_star:.2f}",
    )

    plt.yticks(range(1, 8), ordered_labels)
    plt.ylim(0.5, 7.5)

    plt.xlabel("Corrected Temperature (°C)")
    plt.ylabel("Thermal Comfort Vote")
    plt.title(f"Thermal Comfort vs Temperature — {label}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {
        "case": label,
        "error_column": error_col,

        "comfortable_deg_per_TCV": m_c,
        "comfortable_TCV_per_deg": 1 / m_c,

        "uncomfortable_deg_per_TCV": m_u,
        "uncomfortable_TCV_per_deg": 1 / m_u,

        "intersection_temperature": temp_star,
        "intersection_TCV": tcv_star,
    }


results = []

results.append(run_case("std_T", "standard deviation"))
results.append(run_case("sem_T", "standard error of the mean"))
results.append(run_case("ci95_T", "95% confidence interval"))

results_df = pd.DataFrame(results)

print("\nFINAL COMPARISON")
print(results_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

N_MC = 30000
RANDOM_SEED = 42

# Use "proper" for statistically correct CI propagation.
# Use "literal" only if you want the 95% CI width itself treated as sigma.
ci95_mode = "proper"   # "proper" or "literal"


# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

summary = summary.sort_values("TCV").reset_index(drop=True).copy()

# 95% CI half-width of the mean temperature
summary["ci95_T"] = t.ppf(0.975, summary["n"] - 1) * summary["sem_T"]

# Neutral included in BOTH sides
comfortable_mask = summary["TCV"].between(1, 4)
uncomfortable_mask = summary["TCV"].between(4, 7)

ordered_labels = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]


# ------------------------------------------------------------
# Core functions
# ------------------------------------------------------------

def weighted_fit_temperature_vs_tcv(tcv, temp, sigma):
    """
    Fit:
        Temperature = m * TCV + b

    m = °C per TCV category
    """
    m, b = np.polyfit(
        tcv,
        temp,
        deg=1,
        w=1 / sigma
    )
    return m, b


def intersection(m1, b1, m2, b2):
    """
    Intersection between:
        T = m1 * TCV + b1
        T = m2 * TCV + b2
    """
    tcv_star = (b2 - b1) / (m1 - m2)
    temp_star = m1 * tcv_star + b1
    return temp_star, tcv_star


def get_sampling_sigma(error_col):
    """
    Defines how much to perturb each mean temperature in Monte Carlo.
    """
    if error_col == "ci95_T" and ci95_mode == "proper":
        # Correct interpretation:
        # ci95_T is a 95% half-width, not a 1-sigma uncertainty.
        # Convert it back to SEM.
        tcrit = t.ppf(0.975, summary["n"] - 1)
        return summary["ci95_T"].to_numpy(float) / tcrit

    else:
        # std_T and sem_T are already sigma-like quantities.
        # ci95_T in "literal" mode is treated as if it were sigma.
        return summary[error_col].to_numpy(float)


def run_case(error_col, label):
    rng = np.random.default_rng(RANDOM_SEED)

    tcv_all = summary["TCV"].to_numpy(float)
    temp_mean_all = summary["mean_T"].to_numpy(float)

    # Error bars used in plot and weights used in fit
    sigma_fit_all = summary[error_col].to_numpy(float)

    # Sigma used for Monte Carlo perturbation
    sigma_mc_all = get_sampling_sigma(error_col)

    # Split arrays
    tcv_c = tcv_all[comfortable_mask]
    tcv_u = tcv_all[uncomfortable_mask]

    temp_c_nominal = temp_mean_all[comfortable_mask]
    temp_u_nominal = temp_mean_all[uncomfortable_mask]

    sigma_c_fit = sigma_fit_all[comfortable_mask]
    sigma_u_fit = sigma_fit_all[uncomfortable_mask]

    # --------------------------------------------------------
    # Nominal fit
    # --------------------------------------------------------

    m_c, b_c = weighted_fit_temperature_vs_tcv(
        tcv_c,
        temp_c_nominal,
        sigma_c_fit
    )

    m_u, b_u = weighted_fit_temperature_vs_tcv(
        tcv_u,
        temp_u_nominal,
        sigma_u_fit
    )

    temp_star, tcv_star = intersection(m_c, b_c, m_u, b_u)

    # --------------------------------------------------------
    # Monte Carlo
    # --------------------------------------------------------

    tcv_grid_c = np.linspace(1, 4, 200)
    tcv_grid_u = np.linspace(4, 7, 200)

    temp_lines_c = []
    temp_lines_u = []

    intersections_temp = []
    intersections_tcv = []

    slopes_c = []
    slopes_u = []

    for _ in range(N_MC):
        # Important:
        # perturb all 7 category means once.
        # This means Neutral is the SAME sampled point in both fits.
        sampled_temp_all = rng.normal(temp_mean_all, sigma_mc_all)

        sampled_temp_c = sampled_temp_all[comfortable_mask]
        sampled_temp_u = sampled_temp_all[uncomfortable_mask]

        mc, bc = weighted_fit_temperature_vs_tcv(
            tcv_c,
            sampled_temp_c,
            sigma_c_fit
        )

        mu, bu = weighted_fit_temperature_vs_tcv(
            tcv_u,
            sampled_temp_u,
            sigma_u_fit
        )

        slopes_c.append(mc)
        slopes_u.append(mu)

        temp_lines_c.append(mc * tcv_grid_c + bc)
        temp_lines_u.append(mu * tcv_grid_u + bu)

        if abs(mc - mu) > 1e-12:
            ts, vs = intersection(mc, bc, mu, bu)

            if np.isfinite(ts) and np.isfinite(vs):
                intersections_temp.append(ts)
                intersections_tcv.append(vs)

    temp_lines_c = np.array(temp_lines_c)
    temp_lines_u = np.array(temp_lines_u)

    intersections_temp = np.array(intersections_temp)
    intersections_tcv = np.array(intersections_tcv)

    slopes_c = np.array(slopes_c)
    slopes_u = np.array(slopes_u)

    # 95% shaded bands for fitted lines
    temp_c_low = np.percentile(temp_lines_c, 2.5, axis=0)
    temp_c_mid = np.percentile(temp_lines_c, 50, axis=0)
    temp_c_high = np.percentile(temp_lines_c, 97.5, axis=0)

    temp_u_low = np.percentile(temp_lines_u, 2.5, axis=0)
    temp_u_mid = np.percentile(temp_lines_u, 50, axis=0)
    temp_u_high = np.percentile(temp_lines_u, 97.5, axis=0)

    # Intersection uncertainty
    temp_star_low = np.percentile(intersections_temp, 2.5)
    temp_star_mid = np.percentile(intersections_temp, 50)
    temp_star_high = np.percentile(intersections_temp, 97.5)

    tcv_star_low = np.percentile(intersections_tcv, 2.5)
    tcv_star_mid = np.percentile(intersections_tcv, 50)
    tcv_star_high = np.percentile(intersections_tcv, 97.5)

    # Slope uncertainty
    m_c_low, m_c_mid, m_c_high = np.percentile(slopes_c, [2.5, 50, 97.5])
    m_u_low, m_u_mid, m_u_high = np.percentile(slopes_u, [2.5, 50, 97.5])

    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print(label.upper())
    print("=" * 70)

    print("\nComfortable side: TCV 1–4")
    print(f"Nominal fit: T = {m_c:.5f} * TCV + {b_c:.5f}")
    print(f"°C per TCV = {m_c:.5f}")
    print(f"95% MC range for °C per TCV: {m_c_low:.5f} to {m_c_high:.5f}")

    print("\nUncomfortable side: TCV 4–7")
    print(f"Nominal fit: T = {m_u:.5f} * TCV + {b_u:.5f}")
    print(f"°C per TCV = {m_u:.5f}")
    print(f"95% MC range for °C per TCV: {m_u_low:.5f} to {m_u_high:.5f}")

    print("\nIntersection")
    print(f"Nominal: T = {temp_star:.3f} °C, TCV = {tcv_star:.3f}")
    print(f"MC median: T = {temp_star_mid:.3f} °C, TCV = {tcv_star_mid:.3f}")
    print(f"95% MC range for T: {temp_star_low:.3f} to {temp_star_high:.3f} °C")
    print(f"95% MC range for TCV: {tcv_star_low:.3f} to {tcv_star_high:.3f}")

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plt.figure(figsize=(8, 5))

    # Data points with horizontal error bars
    plt.errorbar(
        summary["mean_T"],
        summary["TCV"],
        xerr=summary[error_col],
        fmt="o",
        capsize=4,
        label=f"Category mean ± {label}",
    )

    # Comfortable-side fit + uncertainty region
    plt.plot(
        temp_c_mid,
        tcv_grid_c,
        label="Fit: Very comfortable → Neutral",
    )

    plt.fill_betweenx(
        tcv_grid_c,
        temp_c_low,
        temp_c_high,
        alpha=0.20,
        label="95% fit uncertainty band",
    )

    # Uncomfortable-side fit + uncertainty region
    plt.plot(
        temp_u_mid,
        tcv_grid_u,
        label="Fit: Neutral → Very uncomfortable",
    )

    plt.fill_betweenx(
        tcv_grid_u,
        temp_u_low,
        temp_u_high,
        alpha=0.20,
    )

    # Intersection point with uncertainty
    plt.errorbar(
        temp_star,
        tcv_star,
        xerr=[
            [temp_star - temp_star_low],
            [temp_star_high - temp_star]
        ],
        yerr=[
            [tcv_star - tcv_star_low],
            [tcv_star_high - tcv_star]
        ],
        fmt="x",
        markersize=10,
        capsize=5,
        label=(
            f"Intersection: {temp_star:.2f} °C, "
            f"TCV={tcv_star:.2f}"
        ),
    )

    plt.yticks(range(1, 8), ordered_labels)
    plt.ylim(0.5, 7.5)

    plt.xlabel("Corrected Temperature (°C)")
    plt.ylabel("Thermal Comfort Vote")
    plt.title(f"Thermal Comfort vs Temperature — {label}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {
        "case": label,
        "error_column": error_col,

        "comfortable_deg_per_TCV_nominal": m_c,
        "comfortable_deg_per_TCV_low95": m_c_low,
        "comfortable_deg_per_TCV_high95": m_c_high,

        "uncomfortable_deg_per_TCV_nominal": m_u,
        "uncomfortable_deg_per_TCV_low95": m_u_low,
        "uncomfortable_deg_per_TCV_high95": m_u_high,

        "intersection_T_nominal": temp_star,
        "intersection_T_median": temp_star_mid,
        "intersection_T_low95": temp_star_low,
        "intersection_T_high95": temp_star_high,

        "intersection_TCV_nominal": tcv_star,
        "intersection_TCV_median": tcv_star_mid,
        "intersection_TCV_low95": tcv_star_low,
        "intersection_TCV_high95": tcv_star_high,
    }


# ------------------------------------------------------------
# Run all three cases
# ------------------------------------------------------------

results = []

results.append(run_case("std_T", "standard deviation"))
results.append(run_case("sem_T", "standard error of the mean"))
results.append(run_case("ci95_T", "95% confidence interval"))

results_df = pd.DataFrame(results)

print("\nFINAL COMPARISON")
print(results_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

N_MC = 30000
RANDOM_SEED = 42

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

summary = summary.sort_values("TCV").reset_index(drop=True).copy()

# 95% CI of the mean temperature
summary["tcrit"] = t.ppf(0.975, summary["n"] - 1)
summary["ci95_T"] = summary["tcrit"] * summary["sem_T"]

# Final convention:
# - SEM is used for fitting and Monte Carlo
# - 95% CI is used for visual error bars
summary["sigma_fit"] = summary["sem_T"]
summary["sigma_mc"] = summary["sem_T"]
summary["xerr_plot"] = summary["ci95_T"]

# Neutral included in BOTH sides
comfortable_mask = summary["TCV"].between(1, 4).to_numpy()
uncomfortable_mask = summary["TCV"].between(4, 7).to_numpy()

ordered_labels = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

# ------------------------------------------------------------
# Functions
# ------------------------------------------------------------

def weighted_fit_temperature_vs_tcv(tcv, temp, sigma):
    """
    Weighted fit:
        Temperature = m * TCV + b

    m = °C per TCV category

    Since uncertainty is in temperature, this is the correct fitting direction.
    """
    m, b = np.polyfit(
        tcv,
        temp,
        deg=1,
        w=1 / sigma
    )
    return m, b


def intersection(m1, b1, m2, b2):
    """
    Intersection of:
        T = m1 * TCV + b1
        T = m2 * TCV + b2
    """
    tcv_star = (b2 - b1) / (m1 - m2)
    temp_star = m1 * tcv_star + b1
    return temp_star, tcv_star


# ------------------------------------------------------------
# Extract arrays
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

tcv_all = summary["TCV"].to_numpy(float)
temp_mean_all = summary["mean_T"].to_numpy(float)

sigma_fit_all = summary["sigma_fit"].to_numpy(float)
sigma_mc_all = summary["sigma_mc"].to_numpy(float)

tcv_c = tcv_all[comfortable_mask]
tcv_u = tcv_all[uncomfortable_mask]

temp_c = temp_mean_all[comfortable_mask]
temp_u = temp_mean_all[uncomfortable_mask]

sigma_c = sigma_fit_all[comfortable_mask]
sigma_u = sigma_fit_all[uncomfortable_mask]

# ------------------------------------------------------------
# Nominal weighted fits
# ------------------------------------------------------------

m_c, b_c = weighted_fit_temperature_vs_tcv(tcv_c, temp_c, sigma_c)
m_u, b_u = weighted_fit_temperature_vs_tcv(tcv_u, temp_u, sigma_u)

temp_star, tcv_star = intersection(m_c, b_c, m_u, b_u)

# ------------------------------------------------------------
# Monte Carlo uncertainty propagation
# ------------------------------------------------------------

tcv_grid_c = np.linspace(1, 4, 250)
tcv_grid_u = np.linspace(4, 7, 250)

temp_lines_c = []
temp_lines_u = []

intersection_T_samples = []
intersection_TCV_samples = []

slope_c_samples = []
slope_u_samples = []

for _ in range(N_MC):
    # Sample all category means once.
    # Neutral is shared by both fits.
    sampled_temp_all = rng.normal(temp_mean_all, sigma_mc_all)

    sampled_temp_c = sampled_temp_all[comfortable_mask]
    sampled_temp_u = sampled_temp_all[uncomfortable_mask]

    mc, bc = weighted_fit_temperature_vs_tcv(tcv_c, sampled_temp_c, sigma_c)
    mu, bu = weighted_fit_temperature_vs_tcv(tcv_u, sampled_temp_u, sigma_u)

    slope_c_samples.append(mc)
    slope_u_samples.append(mu)

    temp_lines_c.append(mc * tcv_grid_c + bc)
    temp_lines_u.append(mu * tcv_grid_u + bu)

    if abs(mc - mu) > 1e-12:
        ts, vs = intersection(mc, bc, mu, bu)

        if np.isfinite(ts) and np.isfinite(vs):
            intersection_T_samples.append(ts)
            intersection_TCV_samples.append(vs)

temp_lines_c = np.array(temp_lines_c)
temp_lines_u = np.array(temp_lines_u)

intersection_T_samples = np.array(intersection_T_samples)
intersection_TCV_samples = np.array(intersection_TCV_samples)

slope_c_samples = np.array(slope_c_samples)
slope_u_samples = np.array(slope_u_samples)

# 95% confidence bands of fitted mean trends
temp_c_low, temp_c_mid, temp_c_high = np.percentile(
    temp_lines_c, [2.5, 50, 97.5], axis=0
)

temp_u_low, temp_u_mid, temp_u_high = np.percentile(
    temp_lines_u, [2.5, 50, 97.5], axis=0
)

# Intersection uncertainty
temp_star_low, temp_star_mid, temp_star_high = np.percentile(
    intersection_T_samples, [2.5, 50, 97.5]
)

tcv_star_low, tcv_star_mid, tcv_star_high = np.percentile(
    intersection_TCV_samples, [2.5, 50, 97.5]
)

# Slope uncertainty
m_c_low, m_c_mid, m_c_high = np.percentile(
    slope_c_samples, [2.5, 50, 97.5]
)

m_u_low, m_u_mid, m_u_high = np.percentile(
    slope_u_samples, [2.5, 50, 97.5]
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 70)
print("FINAL MODEL: 95% CI ERROR BARS, SEM-WEIGHTED FIT")
print("=" * 70)

print("\nComfortable side: TCV 1–4")
print(f"Nominal fit: T = {m_c:.5f} * TCV + {b_c:.5f}")
print(f"°C per TCV = {m_c:.5f}")
print(f"95% MC range for °C per TCV: {m_c_low:.5f} to {m_c_high:.5f}")
print(f"TCV per °C = {1 / m_c:.5f}")

print("\nUncomfortable side: TCV 4–7")
print(f"Nominal fit: T = {m_u:.5f} * TCV + {b_u:.5f}")
print(f"°C per TCV = {m_u:.5f}")
print(f"95% MC range for °C per TCV: {m_u_low:.5f} to {m_u_high:.5f}")
print(f"TCV per °C = {1 / m_u:.5f}")

print("\nIntersection")
print(f"Nominal: T = {temp_star:.3f} °C, TCV = {tcv_star:.3f}")
print(f"MC median: T = {temp_star_mid:.3f} °C, TCV = {tcv_star_mid:.3f}")
print(f"95% MC range for T: {temp_star_low:.3f} to {temp_star_high:.3f} °C")
print(f"95% MC range for TCV: {tcv_star_low:.3f} to {tcv_star_high:.3f}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(8.5, 5.5))

# Points with 95% CI error bars
plt.errorbar(
    summary["mean_T"],
    summary["TCV"],
    xerr=summary["xerr_plot"],
    fmt="o",
    capsize=4,
    label="Category mean ± 95% CI",
)

# Comfortable-side fitted line and confidence band
plt.plot(
    temp_c_mid,
    tcv_grid_c,
    linewidth=2,
    label="Fit: Very comfortable → Neutral",
)

plt.fill_betweenx(
    tcv_grid_c,
    temp_c_low,
    temp_c_high,
    alpha=0.25,
    label="95% confidence band of fitted mean trend",
)

# Uncomfortable-side fitted line and confidence band
plt.plot(
    temp_u_mid,
    tcv_grid_u,
    linewidth=2,
    label="Fit: Neutral → Very uncomfortable",
)

plt.fill_betweenx(
    tcv_grid_u,
    temp_u_low,
    temp_u_high,
    alpha=0.25,
)

# Intersection with propagated uncertainty
plt.errorbar(
    temp_star,
    tcv_star,
    xerr=[
        [temp_star - temp_star_low],
        [temp_star_high - temp_star],
    ],
    yerr=[
        [tcv_star - tcv_star_low],
        [tcv_star_high - tcv_star],
    ],
    fmt="x",
    markersize=11,
    capsize=5,
    label=f"Intersection: {temp_star:.2f} °C, TCV={tcv_star:.2f}",
)

plt.yticks(range(1, 8), ordered_labels)
plt.ylim(0.5, 7.5)

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Thermal Comfort Vote")
plt.title("Thermal Comfort vs Corrected Temperature")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

N_MC = 30000
RANDOM_SEED = 42

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

summary = summary.sort_values("TCV").reset_index(drop=True).copy()

# 95% CI half-width of each category mean
summary["ci95_T"] = t.ppf(0.975, summary["n"] - 1) * summary["sem_T"]

# Convert 95% CI back to 1-sigma uncertainty of the mean
# This is what we use for Monte Carlo sampling
summary["sigma_mean_T"] = summary["ci95_T"] / t.ppf(0.975, summary["n"] - 1)

# Neutral included in BOTH sides
comfortable = summary[summary["TCV"].between(1, 4)].copy()
uncomfortable = summary[summary["TCV"].between(4, 7)].copy()

ordered_labels = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

# ------------------------------------------------------------
# Functions
# ------------------------------------------------------------

def weighted_fit_temperature_vs_tcv(data):
    """
    Fits:
        Temperature = m * TCV + b

    m = °C per TCV category
    """
    tcv = data["TCV"].to_numpy(float)
    temp = data["mean_T"].to_numpy(float)
    sigma = data["sigma_mean_T"].to_numpy(float)

    m, b = np.polyfit(
        tcv,
        temp,
        deg=1,
        w=1 / sigma
    )

    return m, b


def weighted_fit_arrays(tcv, temp, sigma):
    m, b = np.polyfit(
        tcv,
        temp,
        deg=1,
        w=1 / sigma
    )
    return m, b


def intersection(m1, b1, m2, b2):
    """
    Intersection between:
        T = m1 * TCV + b1
        T = m2 * TCV + b2
    """
    tcv_star = (b2 - b1) / (m1 - m2)
    temp_star = m1 * tcv_star + b1
    return temp_star, tcv_star


# ------------------------------------------------------------
# Nominal fits
# ------------------------------------------------------------

m_c, b_c = weighted_fit_temperature_vs_tcv(comfortable)
m_u, b_u = weighted_fit_temperature_vs_tcv(uncomfortable)

temp_star, tcv_star = intersection(m_c, b_c, m_u, b_u)

print("95% CI case")
print("-----------")

print("\nComfortable side: TCV 1–4")
print(f"T = {m_c:.5f} * TCV + {b_c:.5f}")
print(f"°C per TCV = {m_c:.5f}")
print(f"TCV per °C = {1 / m_c:.5f}")

print("\nUncomfortable side: TCV 4–7")
print(f"T = {m_u:.5f} * TCV + {b_u:.5f}")
print(f"°C per TCV = {m_u:.5f}")
print(f"TCV per °C = {1 / m_u:.5f}")

print("\nNominal intersection")
print(f"T = {temp_star:.3f} °C")
print(f"TCV = {tcv_star:.3f}")

# ------------------------------------------------------------
# Monte Carlo uncertainty propagation
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

tcv_all = summary["TCV"].to_numpy(float)
temp_all = summary["mean_T"].to_numpy(float)
sigma_all = summary["sigma_mean_T"].to_numpy(float)

comfortable_mask = summary["TCV"].between(1, 4).to_numpy()
uncomfortable_mask = summary["TCV"].between(4, 7).to_numpy()

tcv_c = tcv_all[comfortable_mask]
tcv_u = tcv_all[uncomfortable_mask]

sigma_c = sigma_all[comfortable_mask]
sigma_u = sigma_all[uncomfortable_mask]

# TCV grids for plotting
tcv_grid_c = np.linspace(1, 4, 250)
tcv_grid_u = np.linspace(4, 7, 250)

# Interpolated uncertainty of category mean along each fitted region
sigma_grid_c = np.interp(tcv_grid_c, tcv_c, sigma_c)
sigma_grid_u = np.interp(tcv_grid_u, tcv_u, sigma_u)

line_samples_c = []
line_samples_u = []

wide_samples_c = []
wide_samples_u = []

intersection_T_samples = []
intersection_TCV_samples = []

slope_c_samples = []
slope_u_samples = []

for _ in range(N_MC):
    # Sample all 7 category means once
    # Neutral is therefore shared by both fits
    sampled_temp_all = rng.normal(temp_all, sigma_all)

    temp_c_sample = sampled_temp_all[comfortable_mask]
    temp_u_sample = sampled_temp_all[uncomfortable_mask]

    mc, bc = weighted_fit_arrays(tcv_c, temp_c_sample, sigma_c)
    mu, bu = weighted_fit_arrays(tcv_u, temp_u_sample, sigma_u)

    slope_c_samples.append(mc)
    slope_u_samples.append(mu)

    line_c = mc * tcv_grid_c + bc
    line_u = mu * tcv_grid_u + bu

    line_samples_c.append(line_c)
    line_samples_u.append(line_u)

    # Wider envelope:
    # possible category-mean values around the fitted line
    wide_c = rng.normal(line_c, sigma_grid_c)
    wide_u = rng.normal(line_u, sigma_grid_u)

    wide_samples_c.append(wide_c)
    wide_samples_u.append(wide_u)

    if abs(mc - mu) > 1e-12:
        ts, vs = intersection(mc, bc, mu, bu)

        if np.isfinite(ts) and np.isfinite(vs):
            intersection_T_samples.append(ts)
            intersection_TCV_samples.append(vs)

line_samples_c = np.array(line_samples_c)
line_samples_u = np.array(line_samples_u)

wide_samples_c = np.array(wide_samples_c)
wide_samples_u = np.array(wide_samples_u)

intersection_T_samples = np.array(intersection_T_samples)
intersection_TCV_samples = np.array(intersection_TCV_samples)

slope_c_samples = np.array(slope_c_samples)
slope_u_samples = np.array(slope_u_samples)

# Narrow band: uncertainty of fitted average line
line_c_low, line_c_mid, line_c_high = np.percentile(
    line_samples_c, [2.5, 50, 97.5], axis=0
)

line_u_low, line_u_mid, line_u_high = np.percentile(
    line_samples_u, [2.5, 50, 97.5], axis=0
)

# Wide band: possible category-mean envelope
wide_c_low, wide_c_mid, wide_c_high = np.percentile(
    wide_samples_c, [2.5, 50, 97.5], axis=0
)

wide_u_low, wide_u_mid, wide_u_high = np.percentile(
    wide_samples_u, [2.5, 50, 97.5], axis=0
)

# Intersection uncertainty
T_low, T_mid, T_high = np.percentile(
    intersection_T_samples, [2.5, 50, 97.5]
)

TCV_low, TCV_mid, TCV_high = np.percentile(
    intersection_TCV_samples, [2.5, 50, 97.5]
)

# Slope uncertainty
mc_low, mc_mid, mc_high = np.percentile(
    slope_c_samples, [2.5, 50, 97.5]
)

mu_low, mu_mid, mu_high = np.percentile(
    slope_u_samples, [2.5, 50, 97.5]
)

print("\nMonte Carlo uncertainty")
print("-----------------------")

print("\nComfortable side °C per TCV")
print(f"Median = {mc_mid:.5f}")
print(f"95% range = {mc_low:.5f} to {mc_high:.5f}")

print("\nUncomfortable side °C per TCV")
print(f"Median = {mu_mid:.5f}")
print(f"95% range = {mu_low:.5f} to {mu_high:.5f}")

print("\nIntersection uncertainty")
print(f"Median T = {T_mid:.3f} °C")
print(f"95% range T = {T_low:.3f} to {T_high:.3f} °C")
print(f"Median TCV = {TCV_mid:.3f}")
print(f"95% range TCV = {TCV_low:.3f} to {TCV_high:.3f}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(9, 5.5))

# 95% CI error bars on category means
plt.errorbar(
    summary["mean_T"],
    summary["TCV"],
    xerr=summary["ci95_T"],
    fmt="o",
    capsize=4,
    label="Category means ± 95% CI",
)

# Comfortable side: wide envelope first
plt.fill_betweenx(
    tcv_grid_c,
    wide_c_low,
    wide_c_high,
    alpha=0.12,
    label="Wide band: possible category-mean envelope",
)

# Comfortable side: narrow fitted-line uncertainty
plt.fill_betweenx(
    tcv_grid_c,
    line_c_low,
    line_c_high,
    alpha=0.30,
    label="Narrow band: fitted-line uncertainty",
)

plt.plot(
    line_c_mid,
    tcv_grid_c,
    linewidth=2,
    label="Fit: Very comfortable → Neutral",
)

# Uncomfortable side: wide envelope
plt.fill_betweenx(
    tcv_grid_u,
    wide_u_low,
    wide_u_high,
    alpha=0.12,
)

# Uncomfortable side: narrow fitted-line uncertainty
plt.fill_betweenx(
    tcv_grid_u,
    line_u_low,
    line_u_high,
    alpha=0.30,
)

plt.plot(
    line_u_mid,
    tcv_grid_u,
    linewidth=2,
    label="Fit: Neutral → Very uncomfortable",
)

# Intersection with uncertainty
plt.errorbar(
    temp_star,
    tcv_star,
    xerr=[
        [temp_star - T_low],
        [T_high - temp_star],
    ],
    yerr=[
        [tcv_star - TCV_low],
        [TCV_high - tcv_star],
    ],
    fmt="x",
    markersize=11,
    capsize=5,
    label=f"Intersection: {temp_star:.2f} °C, TCV={tcv_star:.2f}",
)

plt.yticks(range(1, 8), ordered_labels)
plt.ylim(0.5, 7.5)

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Thermal Comfort Vote")
plt.title("Thermal Comfort vs Temperature — 95% CI only")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

# -----------------------------
# 1. Define the category order manually
# -----------------------------
category_order = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

manual_tcv_map = {label: i + 1 for i, label in enumerate(category_order)}

# -----------------------------
# 2. Prepare data
# -----------------------------
plot_df = df_votes[[x_col, y_col]].dropna().copy()

plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df = plot_df.dropna(subset=[x_col])

# Clean labels
plot_df["comfort_label"] = plot_df[y_col].astype(str).str.strip()

# Use manual order instead of your tcv_map
plot_df["TCV"] = plot_df["comfort_label"].map(manual_tcv_map)

# Keep only known categories
plot_df = plot_df.dropna(subset=["TCV"])
plot_df["TCV"] = plot_df["TCV"].astype(int)

print("Available labels after cleaning:")
print(plot_df["comfort_label"].value_counts())

# -----------------------------
# 3. Summary table
# -----------------------------
summary = (
    plot_df
    .groupby(["comfort_label", "TCV"], as_index=False)[x_col]
    .agg(mean_T="mean", std_T="std", n="count")
    .sort_values("TCV")
    .reset_index(drop=True)
)

summary["sem_T"] = summary["std_T"] / np.sqrt(summary["n"])

print("\nSummary:")
print(summary)

# -----------------------------
# 4. Split exactly as requested
# -----------------------------
lower = summary.iloc[:4].copy()   # first four categories
upper = summary.iloc[4:].copy()   # last three categories

print("\nLower regression data:")
print(lower[["comfort_label", "TCV", "mean_T", "std_T", "n", "sem_T"]])

print("\nUpper regression data:")
print(upper[["comfort_label", "TCV", "mean_T", "std_T", "n", "sem_T"]])

# -----------------------------
# 5. Regression with optional weights
# -----------------------------
def regression_with_ci(data, alpha=0.05):
    data = data.copy()

    # Keep only rows with valid x and y
    data = data[
        np.isfinite(data["TCV"]) &
        np.isfinite(data["mean_T"])
    ]

    if len(data) < 2:
        raise ValueError(
            "There are fewer than 2 valid points. Check the printed summary above."
        )

    x = data["TCV"].to_numpy(dtype=float)
    y = data["mean_T"].to_numpy(dtype=float)

    # Use SEM weights only if they are valid
    if "sem_T" in data.columns and np.all(np.isfinite(data["sem_T"])) and np.all(data["sem_T"] > 0):
        yerr = data["sem_T"].to_numpy(dtype=float)
        w = 1 / yerr**2
        weighted = True
    else:
        w = np.ones_like(y)
        weighted = False

    X = np.column_stack([np.ones_like(x), x])

    XtWX = X.T @ (w[:, None] * X)
    XtWy = X.T @ (w * y)

    cov = np.linalg.pinv(XtWX)
    beta = cov @ XtWy

    intercept, slope = beta

    y_fit = intercept + slope * x
    residuals = y - y_fit

    dof = len(x) - 2

    if dof > 0:
        chi2 = np.sum(w * residuals**2)
        reduced_chi2 = chi2 / dof

        # Scale covariance by residual variance
        cov_scaled = cov * reduced_chi2

        intercept_se, slope_se = np.sqrt(np.diag(cov_scaled))

        tcrit = t.ppf(1 - alpha / 2, dof)

        slope_ci = (
            slope - tcrit * slope_se,
            slope + tcrit * slope_se
        )

        intercept_ci = (
            intercept - tcrit * intercept_se,
            intercept + tcrit * intercept_se
        )
    else:
        reduced_chi2 = np.nan
        cov_scaled = cov
        intercept_se = np.nan
        slope_se = np.nan
        tcrit = np.nan
        slope_ci = (np.nan, np.nan)
        intercept_ci = (np.nan, np.nan)

    return {
        "intercept": intercept,
        "slope": slope,
        "intercept_se": intercept_se,
        "slope_se": slope_se,
        "slope_ci": slope_ci,
        "intercept_ci": intercept_ci,
        "cov": cov_scaled,
        "tcrit": tcrit,
        "dof": dof,
        "reduced_chi2": reduced_chi2,
        "weighted": weighted,
    }


def fitted_line_ci(x_grid, fit):
    x_grid = np.asarray(x_grid, dtype=float)
    y_pred = fit["intercept"] + fit["slope"] * x_grid

    if not np.isfinite(fit["tcrit"]):
        return y_pred, None, None

    X_grid = np.column_stack([np.ones_like(x_grid), x_grid])

    pred_var = np.array([
        row @ fit["cov"] @ row.T
        for row in X_grid
    ])

    pred_se = np.sqrt(pred_var)

    ci_low = y_pred - fit["tcrit"] * pred_se
    ci_high = y_pred + fit["tcrit"] * pred_se

    return y_pred, ci_low, ci_high


lower_fit = regression_with_ci(lower)
upper_fit = regression_with_ci(upper)

# -----------------------------
# 6. Print results
# -----------------------------
print("\nLower regression: first 4 categories")
print("Weighted by SEM:", lower_fit["weighted"])
print(f"Slope = {lower_fit['slope']:.4f} ± {lower_fit['slope_se']:.4f} °C/TCV")
print(
    f"95% CI slope = "
    f"[{lower_fit['slope_ci'][0]:.4f}, {lower_fit['slope_ci'][1]:.4f}] °C/TCV"
)
print(f"Intercept = {lower_fit['intercept']:.4f} °C")
print(f"Reduced chi² = {lower_fit['reduced_chi2']:.4f}")

print("\nUpper regression: last 3 categories")
print("Weighted by SEM:", upper_fit["weighted"])
print(f"Slope = {upper_fit['slope']:.4f} ± {upper_fit['slope_se']:.4f} °C/TCV")
print(
    f"95% CI slope = "
    f"[{upper_fit['slope_ci'][0]:.4f}, {upper_fit['slope_ci'][1]:.4f}] °C/TCV"
)
print(f"Intercept = {upper_fit['intercept']:.4f} °C")
print(f"Reduced chi² = {upper_fit['reduced_chi2']:.4f}")

# -----------------------------
# 7. Plot
# -----------------------------
plt.figure(figsize=(8, 5.5))

plt.errorbar(
    summary["mean_T"],
    summary["TCV"],
    xerr=summary["sem_T"],
    fmt="o",
    capsize=4,
    label="Grouped means ± SEM"
)

# Lower fit
x_lower = np.linspace(lower["TCV"].min(), lower["TCV"].max(), 100)
y_lower, y_lower_low, y_lower_high = fitted_line_ci(x_lower, lower_fit)

plt.plot(
    y_lower,
    x_lower,
    linewidth=2,
    label=(
        f"First 4 fit: slope = "
        f"{lower_fit['slope']:.3f} ± {lower_fit['slope_se']:.3f} °C/TCV"
    )
)

if y_lower_low is not None:
    plt.fill_betweenx(
        x_lower,
        y_lower_low,
        y_lower_high,
        alpha=0.20,
        label="95% CI, first 4 fit"
    )

# Upper fit
x_upper = np.linspace(upper["TCV"].min(), upper["TCV"].max(), 100)
y_upper, y_upper_low, y_upper_high = fitted_line_ci(x_upper, upper_fit)

plt.plot(
    y_upper,
    x_upper,
    linewidth=2,
    label=(
        f"Last 3 fit: slope = "
        f"{upper_fit['slope']:.3f} ± {upper_fit['slope_se']:.3f} °C/TCV"
    )
)

if y_upper_low is not None:
    plt.fill_betweenx(
        x_upper,
        y_upper_low,
        y_upper_high,
        alpha=0.20,
        label="95% CI, last 3 fit"
    )

plt.yticks(range(1, 8), category_order)
plt.ylim(0.5, 7.5)

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Thermal Comfort")
plt.title("Piecewise Regression: Thermal Comfort Vote vs Corrected Temperature")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Base columns
# ----------------------------
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"
gender_col = "gender"


# ----------------------------
# Prepare dataframe
# ----------------------------
plot_df = df_votes[[x_col, y_col, gender_col]].dropna().copy()
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=[x_col, "TCV"]).copy()
plot_df["TCV"] = plot_df["TCV"].astype(int)

# ----------------------------
# Summaries by gender
# ----------------------------
def summarize_by_gender(df_in: pd.DataFrame, gender: str) -> pd.DataFrame:
    s = (
        df_in[df_in[gender_col] == gender]
        .groupby(["TCV"], as_index=False)[x_col]
        .agg(mean_T="mean", std_T="std", n="count")
        .sort_values("TCV")
    )
    s["sem_T"] = s["std_T"] / np.sqrt(s["n"])
    s["ci95_T"] = 1.95 * s["sem_T"]  # keep your 1.95 factor
    s["gender"] = gender
    return s

genders = ["Woman", "Man"]  # change to plot_df[gender_col].dropna().unique() for all
summaries = []
for g in genders:
    if (plot_df[gender_col] == g).any():
        summaries.append(summarize_by_gender(plot_df, g))

summary_all = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
print(summary_all)

# ----------------------------
# Plot: mean temperature vs TSV (3-level) with 95% CI, by gender
# ----------------------------
plt.figure(figsize=(7, 5))

for g in genders:
    s = summary_all[summary_all["gender"] == g]
    if s.empty:
        continue
    plt.errorbar(
        s["mean_T"],
        s["TCV"],
        xerr=s["sem_T"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.yticks([1, 2, 3, 4, 5, 6, 7], ["Very comfortable", "Comfortable", "Slightly comfortable", "Neutral", "Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TCV")
plt.title("Temperature vs grouped TCV, by gender")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))

for g in genders:
    s = summary_all[summary_all["gender"] == g]
    if s.empty:
        continue
    plt.errorbar(
        s["mean_T"],
        s["TCV"],
        xerr=s["ci95_T"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.yticks([1, 2, 3, 4, 5, 6, 7], ["Very comfortable", "Comfortable", "Slightly comfortable", "Neutral", "Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TCV")
plt.title("Temperature vs grouped TCV (95% CI), by gender")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Probem amb tres grups i després també podem fer-ho amb TCV

In [ ]:
df_votes = pd.read_csv(r"../../data/all_surveys(votes).csv")


# choose the two columns
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 1,
    "Slightly comfortable": 2,
    "Neutral":3,
    "Slightly uncomfortable": 4,
    "Uncomfortable": 5,
    "Very uncomfortable": 5,
}



# keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# map thermal sensation to numeric TSV
plot_df["TCV"] = plot_df[y_col].map(tcv_map)

# group by sensation / TSV and compute temperature statistics
summary = (
    plot_df.groupby(["TCV"], as_index=False)[x_col]
    .agg(mean_T="mean", std_T="std", n="count")
    .sort_values("TCV")
)

# optional: standard error instead of standard deviation
summary["sem_T"] = summary["std_T"] / summary["n"]**0.5

print(summary)

# Plot: mean temperature vs mean TSV category, with horizontal error bars in T
plt.figure(figsize=(7, 5))
plt.errorbar(
    summary["mean_T"],          # x = mean temperature
    summary["TCV"],             # y = thermal sensation vote
    xerr=1.95*summary["sem_T"],      # or use summary["std_T"]
    fmt="o",
    capsize=4
)

summary
plt.yticks([1, 2, 3, 4, 5], ["Very Comfortable", "Comfortable", "Neutral", "Uncomfortable", "Very Uncomfortable"])
plt.xlabel("Corrected Temperature (°C)")
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Mean TCV")
plt.title("Temperature vs Thermal Comfort ")
plt.grid(True, alpha=0.3)
plt.show()

# Aquí potser es podria intentar fer un ajust per dues rectes amb dos pendents diferents. (Transició de fase)

In [ ]:
# ----------------------------
# Base columns
# ----------------------------
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"
gender_col = "gender"


tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 1,
    "Slightly comfortable": 2,
    "Neutral":3,
    "Slightly uncomfortable": 4,
    "Uncomfortable": 5,
    "Very uncomfortable": 5,
}


# ----------------------------
# Prepare dataframe
# ----------------------------
plot_df = df_votes[[x_col, y_col, gender_col]].dropna().copy()
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=[x_col, "TCV"]).copy()
plot_df["TCV"] = plot_df["TCV"].astype(int)

# ----------------------------
# Summaries by gender
# ----------------------------
def summarize_by_gender(df_in: pd.DataFrame, gender: str) -> pd.DataFrame:
    s = (
        df_in[df_in[gender_col] == gender]
        .groupby(["TCV"], as_index=False)[x_col]
        .agg(mean_T="mean", std_T="std", n="count")
        .sort_values("TCV")
    )
    s["sem_T"] = s["std_T"] / np.sqrt(s["n"])
    s["ci95_T"] = 1.95 * s["sem_T"]  # keep your 1.95 factor
    s["gender"] = gender
    return s

genders = ["Woman", "Man"]  # change to plot_df[gender_col].dropna().unique() for all
summaries = []
for g in genders:
    if (plot_df[gender_col] == g).any():
        summaries.append(summarize_by_gender(plot_df, g))

summary_all = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
print(summary_all)

# ----------------------------
# Plot: mean temperature vs TSV (3-level) with 95% CI, by gender
# ----------------------------
plt.figure(figsize=(7, 5))

for g in genders:
    s = summary_all[summary_all["gender"] == g]
    if s.empty:
        continue
    plt.errorbar(
        s["mean_T"],
        s["TCV"],
        xerr=s["sem_T"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.yticks([1, 2, 3, 4, 5], ["Very Comfortable", "Comfortable", "Neutral", "Uncomfortable", "Very Uncomfortable"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TCV (grouped)")
plt.title("Temperature vs grouped TCV (sem), by gender")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Base columns
# ----------------------------
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"
gender_col = "gender"


tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 1,
    "Slightly comfortable": 2,
    "Neutral":2,
    "Slightly uncomfortable": 3,
    "Uncomfortable": 3,
    "Very uncomfortable": 3,
}


# ----------------------------
# Prepare dataframe
# ----------------------------
plot_df = df_votes[[x_col, y_col, gender_col]].dropna().copy()
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=[x_col, "TCV"]).copy()
plot_df["TCV"] = plot_df["TCV"].astype(int)

# ----------------------------
# Summaries by gender
# ----------------------------
def summarize_by_gender(df_in: pd.DataFrame, gender: str) -> pd.DataFrame:
    s = (
        df_in[df_in[gender_col] == gender]
        .groupby(["TCV"], as_index=False)[x_col]
        .agg(mean_T="mean", std_T="std", n="count")
        .sort_values("TCV")
    )
    s["sem_T"] = s["std_T"] / np.sqrt(s["n"])
    s["ci95_T"] = 1.95 * s["sem_T"]  # keep your 1.95 factor
    s["gender"] = gender
    return s

genders = ["Woman", "Man"]  # change to plot_df[gender_col].dropna().unique() for all
summaries = []
for g in genders:
    if (plot_df[gender_col] == g).any():
        summaries.append(summarize_by_gender(plot_df, g))

summary_all = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
print(summary_all)

# ----------------------------
# Plot: mean temperature vs TSV (3-level) with 95% CI, by gender
# ----------------------------
plt.figure(figsize=(7, 5))

for g in genders:
    s = summary_all[summary_all["gender"] == g]
    if s.empty:
        continue
    plt.errorbar(
        s["mean_T"],
        s["TCV"],
        xerr=s["sem_T"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.yticks([1, 2, 3], ["Very Comfortable", "Comfortable", "Neutral"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TCV (grouped)")
plt.title("Temperature vs grouped TCV (sem), by gender")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.errorbar(summary["mean_T"], summary["TCV"], xerr=summary["sem_T"], fmt="o", capsize=4)
plt.yticks([1, 2, 3,4,5], ["Very Comfortable", "Comfortable", "Neutral", "Uncomfortable", "Very Uncomfortable"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TCV")
plt.title("Mean temperature with SEM")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7, 5))
plt.errorbar(summary["mean_T"], summary["TCV"], xerr=summary["std_T"], fmt="o", capsize=4)
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TCV")
plt.title("Mean temperature with SD")
plt.grid(True, alpha=0.3)
plt.show()

## Nem a mirar lo de fer $\frac{\Delta T}{T}$ k és igual a $\Delta T$

Mirem fent la relativa respecte el total i respecte la mitjana de cada caminada (aquesta te més sentit per a veure)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

# ----------------------------
# Prep + Kelvin temperature
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["T_K"] = df[temp_col] + 273.15

# ----------------------------
# Relative temperatures (ONLY relatives)
# (use the standard definition: (T - mean)/mean
# ----------------------------
df["T_mean_walk_K"] = df.groupby("walk")["T_K"].transform("mean")
df["relative_temperature_walk_K"] = (df["T_K"] - df["T_mean_walk_K"])

T_mean_all_K = df["T_K"].mean(skipna=True)
df["relative_temperature_all_K"] = (df["T_K"] - T_mean_all_K) 

# ----------------------------
# Plotting dataframe
# ----------------------------
plot_df = df[["relative_temperature_walk_K", "relative_temperature_all_K", y_col]].dropna().copy()
plot_df["TCV"] = plot_df[y_col].map(tcv_map)
plot_df = plot_df.dropna(subset=["TCV"]).copy()
plot_df["TCV"] = plot_df["TCV"].astype(int)

y_ticks = list(range(1, 8))
y_labels = [k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])]

def make_summary(df_in: pd.DataFrame, x_rel_col: str) -> pd.DataFrame:
    s = (
        df_in.groupby("TCV", as_index=False)[x_rel_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("TCV")
    )
    s["sem_x"] = s["std_x"] / np.sqrt(s["n"])
    return s

def plot_total_and_mean(df_in: pd.DataFrame, x_rel_col: str, title_prefix: str, x_label: str):
    # Total scatter
    plt.figure(figsize=(7, 5))
    plt.scatter(df_in[x_rel_col], df_in["TCV"], alpha=0.35)
    plt.yticks(y_ticks, y_labels)
    plt.ylim(0.5, 7.5)
    plt.xlabel(x_label)
    plt.ylabel("Thermal Sensation")
    plt.title(f"{title_prefix} — total")
    plt.grid(True, alpha=0.3)
    plt.show()

    # Mean per TSV with SEM (horizontal errorbars)
    summary = make_summary(df_in, x_rel_col)

    plt.figure(figsize=(7, 5))
    plt.errorbar(
        summary["mean_x"],
        summary["TCV"],
        xerr=summary["sem_x"],
        fmt="o",
        capsize=4,
    )
    plt.yticks(y_ticks, y_labels)
    plt.ylim(0.5, 7.5)
    plt.xlabel(x_label)
    plt.ylabel("Thermal Comfort")
    plt.title(f"{title_prefix} — mean ± SEM")
    plt.grid(True, alpha=0.3)
    plt.show()

    return summary

# ----------------------------
# Plot both: total + mean (like before)
# ----------------------------
summary_walk = plot_total_and_mean(
    plot_df,
    "relative_temperature_walk_K",
    "TCV vs centered temperature (scaled by walk mean)",
    "Relative temperature: (T_K - mean_walk)",
)

summary_all = plot_total_and_mean(
    plot_df,
    "relative_temperature_all_K",
    "TCV vs centered temperature (scaled by global mean)",
    "Relative temperature: (T_K - mean_all)",
)

print("=== Summary (relative by walk) ===")
print(summary_walk)

print("\n=== Summary (relative by all) ===")
print(summary_all)

# Sembla que ara el que ens dona millors observacions (semblen més clares) és el centered by all

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"
gender_col = "gender"

# ----------------------------
# Prep + Kelvin temperature (centered by walk mean)
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["T_K"] = df[temp_col] + 273.15


T_mean_all_K = df["T_K"].mean(skipna=True)
df["relative_temperature_all_K"] = (df["T_K"] - T_mean_all_K) 

df["TCV"] = df[y_col].map(tcv_map)

plot_df = df[["relative_temperature_all_K", "TCV", gender_col]].dropna().copy()
plot_df["TCV"] = plot_df["TCV"].astype(int)

y_ticks = list(range(1, 8))
y_labels = [k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])]

def make_summary(df_in: pd.DataFrame, x_col: str) -> pd.DataFrame:
    s = (
        df_in.groupby("TCV", as_index=False)[x_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("TCV")
    )
    s["sem_x"] = s["std_x"] / np.sqrt(s["n"])
    return s

# ----------------------------
# Plot: mean centered temperature for each TCV, by gender
# (ONLY the means plot)
# ----------------------------
plt.figure(figsize=(8, 5))

for g in ["Woman", "Man"]:  # change to plot_df[gender_col].unique() for all
    sub = plot_df[plot_df[gender_col] == g].copy()
    if sub.empty:
        continue

    summary = make_summary(sub, "relative_temperature_all_K")
    plt.errorbar(
        summary["mean_x"],
        summary["TCV"],
        xerr=summary["sem_x"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.axvline(0, linewidth=1, alpha=0.5)
plt.yticks(y_ticks, y_labels)
plt.ylim(0.5, 7.5)
plt.xlabel("Temperature centered by all walks (K)")
plt.ylabel("Thermal Comfort")
plt.title("Mean centered temperature by TCV, split by gender (± SEM)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 1. em sembla divertit que les dones hagin clavat el neutral en el 0.0

# 2. Sembla un altre cop que comfortable / very comfortable son categories estranyes o que podrien venir donades per alters coses que no fossin la temperatura.

Mirem vots per categoria

In [ ]:
import pandas as pd

y_col = "thermal_comfort"
gender_col = "gender"

target = ["Very comfortable", "Comfortable","Neutral", "Uncomfortable", "Very uncomfortable"]

counts = (
    df_votes[df_votes[gender_col].isin(["Woman", "Man"]) & df_votes[y_col].isin(target)]
    .groupby([gender_col, y_col])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=target, fill_value=0)
)

print(counts)

tenim una mostra considerablement més petita per als valors límit

In [ ]:
import numpy as np
import pandas as pd

temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"
gender_col = "gender"
sun_col = "sun"
wind_col = "wind"

wind_map = {
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
}

sun_map = {
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
}

target_cats = ["Very comfortable", "Comfortable","Slightly comfortable", "Neutral", "Uncomfortable", "Very uncomfortable"]

df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df[sun_col].map(sun_map)
df["wind_num"] = df[wind_col].map(wind_map)

df = df[df[gender_col].isin(["Woman", "Man"]) & df[y_col].isin(target_cats)].copy()
df = df.dropna(subset=[temp_col, "sun_num", "wind_num", gender_col, y_col])

summary = (
    df.groupby([gender_col, y_col], as_index=False)
      .agg(
          n=("ID", "count") if "ID" in df.columns else (temp_col, "count"),
          mean_T=(temp_col, "mean"),
          mean_sun=("sun_num", "mean"),
          mean_wind=("wind_num", "mean"),
      )
)

# nicer ordering
summary[y_col] = pd.Categorical(summary[y_col], categories=target_cats, ordered=True)
summary = summary.sort_values([y_col, gender_col])

print(summary)

# Optional: show Woman-Man differences within each category
pivot = summary.pivot(index=y_col, columns=gender_col, values=["mean_T", "mean_sun", "mean_wind", "n"])
if ("Woman" in pivot["mean_T"].columns) and ("Man" in pivot["mean_T"].columns):
    diffs = pd.DataFrame({
        "ΔT_Woman-Man": pivot["mean_T"]["Woman"] - pivot["mean_T"]["Man"],
        "Δsun_Woman-Man": pivot["mean_sun"]["Woman"] - pivot["mean_sun"]["Man"],
        "Δwind_Woman-Man": pivot["mean_wind"]["Woman"] - pivot["mean_wind"]["Man"],
        "n_Woman": pivot["n"]["Woman"],
        "n_Man": pivot["n"]["Man"],
    })
    print("\n=== Woman - Man differences (within category) ===")
    print(diffs)

# Les categories fins/incloent Neutral estan totes en un rang de temperatures diminut
# També sembla que per les categories més diferenciades (una mica) sí que tenim una diferència de mig grau entre homes i dones
### Sembla que la regió uncomfortable és més fàcilment diferenciable
### Potser intentar predir amb tres grups (?)

In [ ]:
import numpy as np
import pandas as pd

temp_col = "<T-T_fixed+<T>>"
gender_col = "gender"
sun_col = "sun"     # change if different
wind_col = "wind"   # change if different

wind_map = {
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
}

sun_map = {
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
}

df = df_votes.copy()

# Numeric conversions / mappings
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df[sun_col].map(sun_map)
df["wind_num"] = df[wind_col].map(wind_map)

# Keep Woman/Man only (edit if you want to include others)
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

# Drop missing needed fields
df = df.dropna(subset=[temp_col, "sun_num", "wind_num", gender_col]).copy()

# =========================================================
# A) Vote-weighted (each row counts)
# =========================================================
vote_weighted = (
    df.groupby(gender_col, as_index=False)
      .agg(
          n_votes=(temp_col, "count"),
          mean_T=(temp_col, "mean"),
          mean_sun=("sun_num", "mean"),
          mean_wind=("wind_num", "mean"),
      )
)

print("=== Vote-weighted averages (each row = 1) ===")
print(vote_weighted)

if set(vote_weighted[gender_col]) >= {"Woman", "Man"}:
    vw = vote_weighted.set_index(gender_col)
    print("\nVote-weighted Woman - Man:")
    print(pd.Series({
        "ΔT (°C)": vw.loc["Woman", "mean_T"] - vw.loc["Man", "mean_T"],
        "Δsun index": vw.loc["Woman", "mean_sun"] - vw.loc["Man", "mean_sun"],
        "Δwind index": vw.loc["Woman", "mean_wind"] - vw.loc["Man", "mean_wind"],
    }))

# =========================================================
# B) Person-weighted (each person counts once) IF you have an id
# =========================================================
# Change this to your real column name if it exists:
possible_ids = [c for c in ["person_id", "participant_id", "user_id", "respondent_id", "id"] if c in df.columns]
person_id_col = possible_ids[0] if possible_ids else None

if person_id_col:
    person_means = (
        df.groupby([person_id_col, gender_col], as_index=False)
          .agg(
              person_mean_T=(temp_col, "mean"),
              person_mean_sun=("sun_num", "mean"),
              person_mean_wind=("wind_num", "mean"),
              n_votes_person=(temp_col, "count"),
          )
    )

    person_weighted = (
        person_means.groupby(gender_col, as_index=False)
          .agg(
              n_people=(person_id_col, "nunique"),
              mean_T=("person_mean_T", "mean"),
              mean_sun=("person_mean_sun", "mean"),
              mean_wind=("person_mean_wind", "mean"),
              median_T=("person_mean_T", "median"),
              median_sun=("person_mean_sun", "median"),
              median_wind=("person_mean_wind", "median"),
          )
    )

    print(f"\n=== Person-weighted averages (each {person_id_col} = 1) ===")
    print(person_weighted)

    if set(person_weighted[gender_col]) >= {"Woman", "Man"}:
        pw = person_weighted.set_index(gender_col)
        print("\nPerson-weighted Woman - Man:")
        print(pd.Series({
            "ΔT (°C)": pw.loc["Woman", "mean_T"] - pw.loc["Man", "mean_T"],
            "Δsun index": pw.loc["Woman", "mean_sun"] - pw.loc["Man", "mean_sun"],
            "Δwind index": pw.loc["Woman", "mean_wind"] - pw.loc["Man", "mean_wind"],
        }))
else:
    print(
        "\nNOTE: No obvious person-id column found, so I can’t do true person-by-person weighting.\n"
        "If you tell me the participant id column name, I’ll plug it in."
    )

## sembla que la temperatura era baixsa però res de l'altre mon respecte al sol i això. simplement suposo k tenien calor. Estaria bé mirar-ho amb el comfort que tenien

# A PARTIR D'ARA VENEN UN MUNT DE CÀLCULS SOBRE SI SOBREN PER L'ANÀLISI LES CATEGORIES EXTREMALS

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

temp_col = "<T-T_fixed+<T>>"
gender_col = "gender"
sens_col = "thermal_comfort"

sun_map = {"In full shade":-1, "In a mixture of sun and shadow":0, "In full sun":1}
wind_map = {"A strong wind":-1, "A moderate wind":-1, "A light wind":0, "A very light wind":0, "It's not windy":1}

df = df_votes.copy()
df["T"] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df["sun"].map(sun_map)
df["wind_num"] = df["wind"].map(wind_map)
df = df.dropna(subset=["T","sun_num","wind_num",gender_col,sens_col]).copy()
df = df[df[gender_col].isin(["Woman","Man"])].copy()

# heat score (simple; you can tune weights)
df["heat_score"] = 0.5*df["T"] + 2*df["sun_num"] + 1*df["wind_num"]

def auc_for_gender(g):
    sub = df[df[gender_col]==g].copy()
    y = (sub[sens_col] == "Very comfortable").astype(int).to_numpy()
    x = sub["heat_score"].to_numpy()
    if y.sum() < 5 or (y==0).sum() < 5:
        return np.nan
    return roc_auc_score(y, x)

print("AUC (Very comfortable vs others) using heat_score:")
print("  Woman:", auc_for_gender("Woman"))
print("  Man:  ", auc_for_gender("Man"))

In [ ]:
import numpy as np
import pandas as pd

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_comfort"

df = df_votes.copy()

# stop_id = first 2 chars (town+walk) + trailing stop digits
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

def stop_level_corr(label: str):
    sub = df.copy()
    sub["is_label"] = (sub[sens_col] == label).astype(int)

    # per stop + gender: fraction of this label
    stop_gender = (
        sub.groupby(["stop_id", gender_col], as_index=False)
           .agg(n=("is_label", "count"), frac=("is_label", "mean"))
    )

    wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="frac").dropna()

    # also keep how much data each stop has for each gender (for filtering)
    n_wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="n").dropna()

    corr = wide["Woman"].corr(wide["Man"])

    return corr, wide, n_wide

# Correlations
corr_vc, wide_vc, n_vc = stop_level_corr("Very comfortable")
corr_cool, wide_cool, n_cool = stop_level_corr("Comfortable")
corr_scomf, wide_scomf, n_scomf = stop_level_corr("Slightly comfortable")
corr_neutral, wide_neutral, n_neutral = stop_level_corr("Neutral")
corr_suncomf, wide_suncomf, n_suncomf = stop_level_corr("Slightly uncomfortable")
corr_uncomf, wide_uncomf, n_uncomf = stop_level_corr("Uncomfortable")
corr_vuncomf, wide_vuncomf, n_vuncomf = stop_level_corr("Very uncomfortable")

print("Correlation across stops (Women vs Men fraction):")
print("  Very comfortable:", corr_vc)
print("  Comfortable:    ", corr_cool)
print("  Slightly comfortable:", corr_scomf)
print("  Neutral:       ", corr_neutral)
print("  Slightly uncomfortable:", corr_suncomf)
print("  Uncomfortable:  ", corr_uncomf)
print("  Very uncomfortable:", corr_vuncomf)

# OPTIONAL: filter out stops with too few votes per gender (stabilizes correlation)
min_n = 5
mask_vc = (n_vc["Woman"] >= min_n) & (n_vc["Man"] >= min_n)
mask_c = (n_cool["Woman"] >= min_n) & (n_cool["Man"] >= min_n)

print(f"\nFiltered correlations (only stops with >= {min_n} votes for EACH gender):")
print("  Very comfortable:", wide_vc.loc[mask_vc, "Woman"].corr(wide_vc.loc[mask_vc, "Man"]))
print("  Comfortable:    ", wide_cool.loc[mask_c, "Woman"].corr(wide_cool.loc[mask_c, "Man"]))
print("  Slightly comfortable:", wide_scomf.loc[mask_c, "Woman"].corr(wide_scomf.loc[mask_c, "Man"]))
print("  Neutral:       ", wide_neutral.loc[mask_c, "Woman"].corr(wide_neutral.loc[mask_c, "Man"]))
print("  Slightly uncomfortable:", wide_suncomf.loc[mask_c, "Woman"].corr(wide_suncomf.loc[mask_c, "Man"]))
print("  Uncomfortable:  ", wide_uncomf.loc[mask_c, "Woman"].corr(wide_uncomf.loc[mask_c, "Man"]))
print("  Very uncomfortable:", wide_vuncomf.loc[mask_c, "Woman"].corr(wide_vuncomf.loc[mask_c, "Man"]))

## Sembla que la majoria de termes tenen correlacions baixes, sobretot els intermitjos. La cosa és que com a mínim els valors promitjos estan millor diferenciats.

In [ ]:

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_comfort"

df = df_votes.copy()
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

def split_half_reliability(label: str, gender: str, min_n: int = 8, seed: int = 42) -> float:
    rng = np.random.default_rng(seed)
    sub = df[df[gender_col] == gender].copy()
    sub["is_label"] = (sub[sens_col] == label).astype(int)

    fracs_a = {}
    fracs_b = {}

    for stop_id, g in sub.groupby("stop_id"):
        n = len(g)
        if n < min_n:
            continue
        idx = np.arange(n)
        rng.shuffle(idx)
        a = g.iloc[idx[: n // 2]]
        b = g.iloc[idx[n // 2 :]]
        fracs_a[stop_id] = a["is_label"].mean()
        fracs_b[stop_id] = b["is_label"].mean()

    if len(fracs_a) < 10:
        return np.nan  # not enough stops to estimate reliably

    s1 = pd.Series(fracs_a)
    s2 = pd.Series(fracs_b).reindex(s1.index)
    return float(s1.corr(s2))

for label in ["Very comfortable", "Comfortable","Slightly comfortable", "Neutral", "Uncomfortable", "Very uncomfortable"]:
    for gender in ["Woman", "Man"]:
        r = split_half_reliability(label, gender, min_n=6, seed=42)
        print(f"Split-half r | label={label:8s} gender={gender:5s}: {r}")

In [ ]:
import numpy as np
import pandas as pd

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_comfort"

df = df_votes.copy()
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

df["is_comfortable"] = df[sens_col].isin(["Very comfortable", "Comfortable"]).astype(int)

# Between-gender corr across stops
stop_gender = (
    df.groupby(["stop_id", gender_col], as_index=False)
      .agg(n=("is_comfortable","count"), frac=("is_comfortable","mean"))
)
wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="frac").dropna()
nwide = stop_gender.pivot(index="stop_id", columns=gender_col, values="n").dropna()

print("Comfortable corr (raw):", wide["Woman"].corr(wide["Man"]))

min_n = 4
mask = (nwide["Woman"]>=min_n) & (nwide["Man"]>=min_n)
print(f"Comfortable corr (>= {min_n} votes each):", wide.loc[mask,"Woman"].corr(wide.loc[mask,"Man"]))

# Within-gender split-half reliability (one random split)
def split_half_r(label_col: str, gender: str, min_n: int = 4, seed: int = 42):
    rng = np.random.default_rng(seed)
    sub = df[df[gender_col] == gender].copy()

    fracs_a, fracs_b = {}, {}
    for stop_id, g in sub.groupby("stop_id"):
        n = len(g)
        if n < min_n:
            continue
        idx = np.arange(n)
        rng.shuffle(idx)
        a = g.iloc[idx[: n // 2]]
        b = g.iloc[idx[n // 2 :]]
        fracs_a[stop_id] = a[label_col].mean()
        fracs_b[stop_id] = b[label_col].mean()

    if len(fracs_a) < 5:
        return np.nan
    s1 = pd.Series(fracs_a)
    s2 = pd.Series(fracs_b).reindex(s1.index)
    return float(s1.corr(s2))

print("Split-half r (Comfortable) Woman:", split_half_r("is_comfortable", "Woman", min_n=4))
print("Split-half r (Comfortable) Man:  ", split_half_r("is_comfortable", "Man", min_n=4))

In [ ]:
import numpy as np
import pandas as pd

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_comfort"

df = df_votes.copy()
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

df["is_supercomfortable"] = df[sens_col].isin(["Very comfortable","Comfortable"]).astype(int)
df["is_comfortableside"] = df[sens_col].isin(["Comfortable", "Very comfortable","Slightly comfortable"]).astype(int)

def between_gender_corr(label_col: str, min_n: int | None = None):
    stop_gender = (
        df.groupby(["stop_id", gender_col], as_index=False)
          .agg(n=(label_col, "count"), frac=(label_col, "mean"))
    )
    wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="frac").dropna()
    if min_n is None:
        return float(wide["Woman"].corr(wide["Man"]))

    nwide = stop_gender.pivot(index="stop_id", columns=gender_col, values="n").dropna()
    mask = (nwide["Woman"] >= min_n) & (nwide["Man"] >= min_n)
    return float(wide.loc[mask, "Woman"].corr(wide.loc[mask, "Man"]))

def split_half_r(label_col: str, gender: str, min_n: int = 4, seed: int = 42):
    rng = np.random.default_rng(seed)
    sub = df[df[gender_col] == gender].copy()

    fracs_a, fracs_b = {}, {}
    for stop_id, g in sub.groupby("stop_id"):
        n = len(g)
        if n < min_n:
            continue
        idx = np.arange(n)
        rng.shuffle(idx)
        a = g.iloc[idx[: n // 2]]
        b = g.iloc[idx[n // 2 :]]
        fracs_a[stop_id] = a[label_col].mean()
        fracs_b[stop_id] = b[label_col].mean()

    if len(fracs_a) < 5:
        return np.nan
    s1 = pd.Series(fracs_a)
    s2 = pd.Series(fracs_b).reindex(s1.index)
    return float(s1.corr(s2))

min_n = 4

for name, col in [("Supercomfortable", "is_supercomfortable"), ("Comfortable-side (Comfortable+Very comfortable+Slightly comfortable)", "is_comfortableside")]:
    print(f"\n=== {name} ===")
    print("Between-gender corr (raw):", between_gender_corr(col, min_n=None))
    print(f"Between-gender corr (>= {min_n} votes each):", between_gender_corr(col, min_n=min_n))
    print(f"Split-half r Woman (min_n={min_n}):", split_half_r(col, "Woman", min_n=min_n))
    print(f"Split-half r Man   (min_n={min_n}):", split_half_r(col, "Man", min_n=min_n))

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import roc_auc_score

temp_col = "<T-T_fixed+<T>>"
sens_col = "thermal_comfort"

sun_map = {"In full shade":-1, "In a mixture of sun and shadow":0, "In full sun":1}
wind_map = {"A strong wind":-1, "A moderate wind":-1, "A light wind":0, "A very light wind":0, "It's not windy":1}

cats = ["Very comfortable","Comfortable","Slightly comfortable","Neutral","Slightly uncomfortable","Uncomfortable","Very uncomfortable"]

df = df_votes.copy()
df["T"] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df["sun"].map(sun_map)
df["wind_num"] = df["wind"].map(wind_map)
df = df.dropna(subset=["T", sens_col]).copy()
df = df[df[sens_col].isin(cats)].copy()

# choose score: temperature only OR heat_score
df["score_T"] = df["T"]
df["score_heat"] = df["T"] + 0.7*df["sun_num"].fillna(0) + 0.7*df["wind_num"].fillna(0)

def pairwise_auc(score_col: str):
    rows = []
    for a, b in combinations(cats, 2):
        sub = df[df[sens_col].isin([a,b])].copy()
        if sub.shape[0] < 30:
            continue
        y = (sub[sens_col] == b).astype(int).to_numpy()
        x = sub[score_col].to_numpy()
        auc = roc_auc_score(y, x)
        rows.append({"cat_a": a, "cat_b": b, "auc": auc, "n": len(sub)})
    out = pd.DataFrame(rows).sort_values("auc")
    return out

print("=== Most-overlapping pairs b y Temperature only (AUC near 0.5) ===")
print(pairwise_auc("score_T").head(15))

print("\n=== Most-overlapping pairs by Heat score (AUC near 0.5) ===")
print(pairwise_auc("score_heat").head(15))

## He testejat amb l'Humidex i bàsicament sembla que solapa més

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.metrics import roc_auc_score

sens_col = "thermal_comfort"

temp_col = "<T-T_fixed+<T>>"
hdx_col  = "<HDX-HDX_fixed+<HDX>>"

cats = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

def compute_overlap_matrix(df, value_col, sens_col, cats, min_n=30):
    work = df.copy()
    work["score"] = pd.to_numeric(work[value_col], errors="coerce")
    work = work.dropna(subset=["score", sens_col]).copy()
    work = work[work[sens_col].isin(cats)].copy()

    auc_mat = pd.DataFrame(np.nan, index=cats, columns=cats, dtype=float)

    for a, b in combinations(cats, 2):
        sub = work[work[sens_col].isin([a, b])].copy()
        if len(sub) < min_n:
            continue

        y = (sub[sens_col] == b).astype(int).to_numpy()
        x = sub["score"].to_numpy()

        if len(np.unique(y)) < 2 or len(np.unique(x)) < 2:
            continue

        auc = roc_auc_score(y, x)
        auc_mat.loc[a, b] = auc
        auc_mat.loc[b, a] = 1 - auc

    overlap_mat = 1 - 2 * (auc_mat - 0.5).abs()
    np.fill_diagonal(overlap_mat.values, 1.0)

    return auc_mat, overlap_mat

def draw_heatmap(ax, mat, title, cmap, vmin=None, vmax=None, fmt=".2f"):
    im = ax.imshow(mat.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index)
    ax.set_title(title)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, format(val, fmt), ha="center", va="center", fontsize=8)

    return im

auc_T, overlap_T = compute_overlap_matrix(df_votes, temp_col, sens_col, cats, min_n=30)
auc_HDX, overlap_HDX = compute_overlap_matrix(df_votes, hdx_col, sens_col, cats, min_n=30)
overlap_diff = overlap_HDX - overlap_T

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

im0 = draw_heatmap(
    axes[0], overlap_T, "Overlap using Temperature", cmap="viridis", vmin=0, vmax=1
)
fig.colorbar(im0, ax=axes[0], label="Overlap")

im1 = draw_heatmap(
    axes[1], overlap_HDX, "Overlap using Humidex", cmap="viridis", vmin=0, vmax=1
)
fig.colorbar(im1, ax=axes[1], label="Overlap")

max_abs = np.nanmax(np.abs(overlap_diff.values))
im2 = draw_heatmap(
    axes[2], overlap_diff, "Difference (HDX - T)", cmap="RdBu_r", vmin=-max_abs, vmax=max_abs
)
fig.colorbar(im2, ax=axes[2], label="Δ overlap")

plt.tight_layout()
plt.show()

## Now we try to separate "COLD/WARM" walks, to see if there are visible differences

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

temp_col = "<T-T_fixed+<T>>"   # o la columna absoluta que vulguis usar per classificar caminades
y_col = "thermal_comfort"

y_ticks = list(range(1, 8))
y_labels = [k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])]

# ----------------------------
# Prepare dataframe
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["TSV"] = df[y_col].map(tcv_map)

df = df.dropna(subset=[temp_col, "walk", "TSV"]).copy()
df["TSV"] = df["TSV"].astype(int)

# ----------------------------
# 1) Global reference across all walks
# ----------------------------
T_mean_all = df[temp_col].mean()
df["relative_temperature_all"] = df[temp_col] - T_mean_all

# ----------------------------
# 2) Classify each walk by its ABSOLUTE mean temperature
# ----------------------------
walk_summary = (
    df.groupby("walk", as_index=False)[temp_col]
    .mean()
    .rename(columns={temp_col: "walk_mean_abs"})
)

walk_summary["walk_group"] = pd.qcut(
    walk_summary["walk_mean_abs"],
    q=2,
    labels=["Cold walks", "Warm walks"]
)

# map back to vote rows
df["walk_group"] = df["walk"].map(walk_summary.set_index("walk")["walk_group"])
df["walk_mean_abs"] = df["walk"].map(walk_summary.set_index("walk")["walk_mean_abs"])

# group summaries
group_avg = (
    walk_summary.groupby("walk_group", as_index=False)["walk_mean_abs"]
    .mean()
    .rename(columns={"walk_mean_abs": "avg_walk_mean_abs"})
)
print("=== Mean absolute temperature of each walk group ===")
print(group_avg)

avg_map = group_avg.set_index("walk_group")["avg_walk_mean_abs"].to_dict()

# ----------------------------
# Summarize using GLOBAL relative temperature
# ----------------------------
def summarize_mean_T_for_TSV(sub_df: pd.DataFrame) -> pd.DataFrame:
    s = (
        sub_df.groupby("TSV", as_index=False)["relative_temperature_all"]
        .agg(mean_T="mean", std_T="std", n="count")
        .sort_values("TSV")
    )

    # SEM only where n > 1 and std exists
    s["sem_T"] = s["std_T"] / np.sqrt(s["n"])
    return s

# ----------------------------
# Plot
# ----------------------------
plt.figure(figsize=(8, 5))

for grp in ["Cold walks", "Warm walks"]:
    sub = df[df["walk_group"] == grp].dropna(subset=["TSV", "relative_temperature_all"])
    if sub.empty:
        continue

    summary = summarize_mean_T_for_TSV(sub)
    if summary.empty:
        continue

    avg_walk_mean = avg_map.get(grp, float("nan"))

    plt.errorbar(
        summary["mean_T"],
        summary["TSV"],
        xerr=summary["sem_T"],
        fmt="o-",
        capsize=4,
        label=f"{grp} (mean abs T = {avg_walk_mean:.2f})",
    )

y_ticks = sorted(tcv_map.values())
y_labels = [k for k, v in sorted(tcv_map.items(), key=lambda kv: kv[1])]
plt.axvline(0, linewidth=1, alpha=0.5)  # 0 = global mean across all walks
plt.xlabel("Temperature relative to global mean")
plt.ylabel("Thermal Comfort (TCV)")
plt.title("Global-relative temperature by comfort, split by globally cold vs warm walks")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Sembla que el pendent és el mateix apx i que no hi ha massa diferències

# Ara anem a mirar mean TSV x T bin

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

# ----------------------------
# Prepare dataframe (no walk split)
# ----------------------------
df = df_votes.copy()

df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["TCV"] = df[y_col].map(tcv_map)
df = df.dropna(subset=[temp_col, "TCV"]).copy()

# ----------------------------
# Make temperature bins
# ----------------------------
n_bins = 10  # change if you want
tmin = df[temp_col].min()
tmax = df[temp_col].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)

df["T_bin"] = pd.cut(df[temp_col], bins=bin_edges, include_lowest=True)

# ----------------------------
# Mean TCV per bin (+ SEM)
# ----------------------------
summary = (
    df.groupby("T_bin", observed=False)
    .agg(
        mean_T=(temp_col, "mean"),
        mean_TCV=("TCV", "mean"),
        std_TCV=("TCV", "std"),
        n=("TCV", "count"),
    )
    .dropna()
    .reset_index()
    .sort_values("mean_T")
)

summary["sem_TCV"] = summary["std_TCV"] / np.sqrt(summary["n"])

print(summary[["T_bin", "mean_T", "mean_TCV", "n", "sem_TCV"]])

# ----------------------------
# Plot: mean TCV vs temperature bin mean
# ----------------------------
plt.figure(figsize=(8, 5))

plt.errorbar(
    summary["mean_T"],
    summary["mean_TCV"],
    yerr=summary["sem_TCV"],
    fmt="o-",
    capsize=4,
)

plt.xlabel("Temperature (°C) (bin mean)")
plt.ylabel("Mean TCV")
plt.title("Mean TCV by temperature bins")
plt.yticks(
    [1, 2, 3, 4, 5, 6, 7],
    [
        "Very comfortable",
        "Comfortable",
        "Slightly comfortable",
        "Neutral",
        "Slightly uncomfortable",
        "Uncomfortable",
        "Very uncomfortable",
    ]
)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## En 3 grups

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 1,
    "Slightly comfortable": 1,
    "Neutral": 1,
    "Slightly uncomfortable": 3,
    "Uncomfortable": 3,
    "Very uncomfortable": 3,
}

# ----------------------------
# Prepare dataframe (no walk split)
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["TCV"] = df[y_col].map(tcv_map)
df = df.dropna(subset=[temp_col, "TCV"]).copy()

# ----------------------------
# Make temperature bins
# ----------------------------
n_bins = 8# change if you want
tmin = df[temp_col].min()
tmax = df[temp_col].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)

df["T_bin"] = pd.cut(df[temp_col], bins=bin_edges, include_lowest=True)

# ----------------------------
# Mean TSV per bin (+ SEM)
# ----------------------------
summary = (
    df.groupby("T_bin", observed=False)
    .agg(
        mean_T=(temp_col, "mean"),
        mean_TCV=("TCV", "mean"),
        std_TCV=("TCV", "std"),
        n=("TCV", "count"),
    )
    .dropna()
    .reset_index()
    .sort_values("mean_T")
)
summary["sem_TCV"] = summary["std_TCV"] / np.sqrt(summary["n"])

print(summary[["T_bin", "mean_T", "mean_TCV", "n", "sem_TCV"]])

plt.figure(figsize=(8, 5))
plt.errorbar(
    summary["mean_T"],
    summary["mean_TCV"],
    yerr=summary["sem_TCV"],
    fmt="o-",
    capsize=4,
)

plt.xlabel("Temperature (°C) (bin mean)")
plt.ylabel("Mean TCV")
plt.title("Mean TCV by temperature bins")
plt.yticks([1, 2, 3])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

tcv_map = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

# ----------------------------
# Prepare dataframe (no walk split)
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["TCV"] = df[y_col].map(tcv_map)
df = df.dropna(subset=[temp_col, "TCV"]).copy()

# ----------------------------
# Make temperature bins
# ----------------------------
n_bins = 8# change if you want
tmin = df[temp_col].min()
tmax = df[temp_col].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)

df["T_bin"] = pd.cut(df[temp_col], bins=bin_edges, include_lowest=True)

# ----------------------------
# Mean TSV per bin (+ SEM)
# ----------------------------
summary = (
    df.groupby("T_bin", observed=False)
    .agg(
        mean_T=(temp_col, "mean"),
        mean_TCV=("TCV", "mean"),
        std_TCV=("TCV", "std"),
        n=("TCV", "count"),
    )
    .dropna()
    .reset_index()
    .sort_values("mean_T")
)
summary["sem_TCV"] = summary["std_TCV"] / np.sqrt(summary["n"])



# ----------------------------
# Plot: mean TCV vs temperature bin mean
# ----------------------------
plt.figure(figsize=(8, 5))
plt.errorbar(
    summary["mean_T"],
    summary["mean_TCV"],
    yerr=summary["sem_TCV"],
    fmt="o-",
    capsize=4,
)

plt.xlabel("Temperature (°C) (bin mean)")
plt.ylabel("Mean TCV")
plt.title("Mean TCV by temperature bins")
plt.yticks([1, 2, 3,4,5,6,7])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"      # corrected temperature column
y_col = "thermal_comfort"
gender_col = "gender"


# ----------------------------
# Prepare dataframe (no walk-centering)
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["TSV"] = df[y_col].map(tcv_map)

df = df.dropna(subset=[temp_col, "TSV", gender_col]).copy()
df["TSV"] = df["TSV"].astype(int)

# ----------------------------
# Shared temperature bins (global) + bin centers
# ----------------------------
n_bins = 12  # tweak if you want
tmin = df[temp_col].min()
tmax = df[temp_col].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

df["T_bin"] = pd.cut(df[temp_col], bins=bin_edges, include_lowest=True)
cats = df["T_bin"].cat.categories
center_map = dict(zip(cats, bin_centers))

def summarize_mean_tsv_by_bin(sub_df: pd.DataFrame) -> pd.DataFrame:
    s = (
        sub_df.groupby("T_bin", observed=False)
        .agg(
            mean_TSV=("TSV", "mean"),
            std_TSV=("TSV", "std"),
            n=("TSV", "count"),
        )
        .dropna()
        .reset_index()
    )
    s["sem_TSV"] = s["std_TSV"] / np.sqrt(s["n"])
    s["x_center"] = s["T_bin"].map(center_map)
    return s.sort_values("x_center")

# ----------------------------
# Plot by gender: mean TSV vs corrected temperature
# ----------------------------
plt.figure(figsize=(8, 5))

for gender in ["Woman", "Man"]:
    sub = df[df[gender_col] == gender].dropna(subset=["T_bin", "TSV"])
    if sub.empty:
        continue

    summary = summarize_mean_tsv_by_bin(sub)
    if summary.empty:
        continue

    plt.errorbar(
        summary["x_center"],
        summary["mean_TSV"],
        yerr=summary["sem_TSV"],
        fmt="o-",
        capsize=4,
        label=gender,
    )

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Mean TSV")
plt.title("Mean TSV vs corrected temperature, split by gender")
plt.yticks([1, 2, 3, 4, 5, 6, 7])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## i ara anem a diferenciar per COLD/WARM caminades

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"

tcv_map = {
    "Slightly cool": 1,
    "Cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}

# ----------------------------
# Prepare dataframe
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["TSV"] = df[y_col].map(tcv_map)

df = df.dropna(subset=[temp_col, "walk", "TSV"]).copy()
df["TSV"] = df["TSV"].astype(int)

# Walk mean temperature (°C) and centered temp (°C)
df["T_mean_walk_C"] = df.groupby("walk")[temp_col].transform("mean")
df["T_centered_walk_C"] = df[temp_col] - df["T_mean_walk_C"]

# ----------------------------
# Walk-level table + Cold/Warm split
# ----------------------------
walk_summary = (
    df.groupby("walk", as_index=False)["T_mean_walk_C"]
    .first()
    .rename(columns={"T_mean_walk_C": "walk_mean_C"})
)

# 2 quantiles => Cold vs Warm (roughly equal number of walks)
walk_summary["walk_group"] = pd.qcut(
    walk_summary["walk_mean_C"],
    q=2,
    labels=["Cold walks", "Warm walks"],
)

df["walk_group"] = df["walk"].map(walk_summary.set_index("walk")["walk_group"])

# ----------------------------
# Print average temperature per group
# (average of walk means; not weighted by #votes)
# ----------------------------
group_avg = walk_summary.groupby("walk_group", as_index=False)["walk_mean_C"].agg(
    avg_walk_mean_C="mean",
    n_walks="count",
)
print("=== Average temperature by group (mean of walk means) ===")
print(group_avg)

# (Optional) also vote-weighted mean temperature, if you want
vote_weighted = (
    df.groupby("walk_group", as_index=False)[temp_col]
    .agg(avg_vote_weighted_T_C="mean", n_votes="count")
)
print("\n=== Vote-weighted average temperature by group ===")
print(vote_weighted)

# ----------------------------
# Shared bins for centered temperature (so curves are comparable)
# ----------------------------
n_bins = 10
tmin = df["T_centered_walk_C"].min()
tmax = df["T_centered_walk_C"].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)
df["T_bin"] = pd.cut(df["T_centered_walk_C"], bins=bin_edges, include_lowest=True)

def summarize_mean_tsv_by_bin(sub_df: pd.DataFrame) -> pd.DataFrame:
    s = (
        sub_df.groupby("T_bin", observed=False)
        .agg(
            mean_T=("T_centered_walk_C", "mean"),
            mean_TSV=("TSV", "mean"),
            std_TSV=("TSV", "std"),
            n=("TSV", "count"),
        )
        .dropna()
        .reset_index()
        .sort_values("mean_T")
    )
    s["sem_TSV"] = s["std_TSV"] / np.sqrt(s["n"])
    return s

# ----------------------------
# Plot: Mean TSV vs centered temperature, Cold vs Warm walks
# ----------------------------
plt.figure(figsize=(8, 5))

for grp in ["Cold walks", "Warm walks"]:
    sub = df[df["walk_group"] == grp].dropna(subset=["T_bin", "TSV", "T_centered_walk_C"])
    if sub.empty:
        continue

    summary = summarize_mean_tsv_by_bin(sub)
    if summary.empty:
        continue

    plt.errorbar(
        summary["mean_T"],
        summary["mean_TSV"],
        yerr=summary["sem_TSV"],
        fmt="o-",
        capsize=4,
        label=f"{grp} (n={int(summary['n'].sum())})",
    )

plt.xlabel("Temperature centered by walk mean (°C)")
plt.ylabel("Mean TSV")
plt.title("Mean TSV vs centered temperature (by-walk), split by cold vs warm walks")
plt.yticks([1, 2, 3, 4, 5, 6, 7])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Sembla que simplement tenim un desplaçament vertical i ben bé k prou més. Els valors no estàn en el mateix x per a tenir prous valors per fer l'average (cold walks i warm walks no tenen les mateixes dades)